# Notebook 1 — The Generator

**Webinar 1: The AI-Ready Data Audit**

---

### Purpose

Generate a synthetic student-performance dataset with **intentional anomalies**
so the downstream notebooks have meaningful defects to detect and remediate.

### Injected Defects

| Defect Type | Column | What We Inject |
|-------------|--------|----------------|
| Missing values | `marks_math`, `attendance_pct` | `NaN` in 3 rows each |
| Impossible value (< floor) | `marks_science` | −10 (marks can't be negative) |
| Impossible value (> ceiling) | `attendance_pct` | 141% (max is 100%) |
| Inconsistent formatting | `city` | 'Bangalore' vs 'Bengaluru', case/whitespace variants |
| Exact duplicate row | full row | 1–2 rows duplicated (Data Leakage trap) |

### Output

`data/student_data.xlsx`

In [ ]:
import os
import pandas as pd
import numpy as np

DATA_DIR = os.path.join("..", "data")
os.makedirs(DATA_DIR, exist_ok=True)

## Build Base Student Records

In [ ]:
np.random.seed(42)
n = 40

# ---------------------------------------------------------------------------
# Teacher notes — unstructured text that a downstream RAG pipeline would
# embed and index.  Varied in length, tone, and vocabulary to make future
# NLP exercises non-trivial.
# ---------------------------------------------------------------------------
TEACHER_NOTES_POOL = [
    "Excellent problem-solver. Consistently finishes assignments ahead of schedule and helps peers.",
    "Struggles with algebra but shows strong effort. Needs additional practice on quadratic equations.",
    "Frequently absent. When present, participates actively but misses too many foundational lessons.",
    "Top performer in science labs. Written exam scores do not reflect practical ability.",
    "Quiet in class but produces high-quality written work. May benefit from oral presentation practice.",
    "Disruptive behavior noted in Q2. Improvement observed after parent-teacher meeting in March.",
    "Strong in both math and science. Recommended for the advanced enrichment program next semester.",
    "Average performance across subjects. No major concerns but could be more engaged in group work.",
    "Exceptional creative writing skills. Below average in quantitative subjects — consider tutoring.",
    "Transferred mid-year from another school. Still adapting to the curriculum pace.",
    "Consistently late to morning classes. Academic performance drops in first-period subjects.",
    "Shows aptitude for data analysis projects. Completed the optional statistics module independently.",
    "Needs support with reading comprehension which impacts performance across all subjects.",
    "Highly motivated self-learner. Asks insightful questions that push classroom discussions forward.",
    "Performance declined after mid-term. Follow-up with school counselor recommended.",
]

data = {
    "student_id": range(1, n + 1),
    "marks_math": np.random.randint(30, 100, n).astype(float),
    "marks_science": np.random.randint(30, 100, n).astype(float),
    "attendance_pct": np.random.randint(55, 100, n).astype(float),
    "city": np.random.choice(
        ["Delhi", "delhi ", "Mumbai", "mumbai",
         "Bangalore", "Bengaluru", "bangalore ", "Jaipur"], n
    ),
    "teacher_notes": np.random.choice(TEACHER_NOTES_POOL, n),
}

df = pd.DataFrame(data)

print("=" * 60)
print("  NOTEBOOK 1 — THE GENERATOR")
print("=" * 60)
print(f"\n✅ Base dataset created: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Columns: {list(df.columns)}")
df.head()

## Inject Realistic Data-Quality Problems

We deliberately introduce issues so the pipeline has something to detect:
- **Missing values** in `marks_math` and `attendance_pct`
- **Impossible values** (attendance > 100%, negative marks)
- **Inconsistent city formatting** already embedded in the generation above
- **Exact duplicate rows** — the Data Leakage trap

In [ ]:
# --- Missing values ---
missing_idx = np.random.choice(df.index, 6, replace=False)
df.loc[missing_idx[:3], "marks_math"] = np.nan
df.loc[missing_idx[3:], "attendance_pct"] = np.nan

# --- Impossible values ---
df.loc[5, "marks_science"] = -10           # marks can't be negative
df.loc[2, "attendance_pct"] = 141           # attendance can't exceed 100%

# --- Exact duplicate rows (Data Leakage trap) ---
df = pd.concat([df, df.iloc[[7, 23]]], ignore_index=True)

print("── Injected Defects ──")
print(f"   Missing marks_math      : {df['marks_math'].isna().sum()}")
print(f"   Missing attendance_pct   : {df['attendance_pct'].isna().sum()}")
print(f"   marks_science < 0        : {(df['marks_science'] < 0).sum()}")
print(f"   attendance_pct > 100     : {(df['attendance_pct'] > 100).sum()}")
print(f"   Exact duplicate rows     : {df.duplicated().sum()}")
print(f"   Unique city spellings    : {df['city'].nunique()} → {sorted(df['city'].unique())}")
print(f"\n   Final shape: {df.shape}")

## Export to Excel

In [ ]:
output_file = os.path.join(DATA_DIR, "student_data.xlsx")
df.to_excel(output_file, index=False, engine="openpyxl")

print(f"✅ Saved: {output_file}")
print(f"   Shape: {df.shape}")
print(f"\n→ Next: Notebook 2 — Data Profiling & The LLM Defense Line")